# Genome-wide Prediction → BigWig
Slide a window across BED regions or a full chromosome and write ES/MS/LS/G1/WRT BigWig tracks.

In [ ]:
import sys
import numpy as np
import torch
import pyBigWig
from pathlib import Path
from pyfaidx import Fasta

PROJECT_ROOT = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path('/Users/lzz/Documents/GitHub/repli-ATAC-seq/Transformer')
sys.path.insert(0, str(PROJECT_ROOT))

from src.infer._utils import load_model, parse_bed, fetch_one_hot, rc_average
from src.data.data_utils import GenomeSequence

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
CHECKPOINT  = '/liaozizhuo/repli-ATAC-seq/outputs/basenji2_wt/checkpoints/best_model.pt'
CONFIG      = str(PROJECT_ROOT / 'src/configs/transformer_wt.yaml')
FASTA       = '/liaozizhuo/repli-ATAC-seq/data/genomes/rice_all_genomes_v7.fasta'
SPECIES     = 'rice'           # must match a name in manifest.yaml

# BED mode: predict for each row in BED file
# Chrom mode: sliding window over a chromosome
MODE        = 'chrom'          # 'bed' | 'chrom'
BED         = 'data/regions.bed'
CHROM       = 'chr12'

BATCH_SIZE  = 32
RC_AVERAGE  = True             # average forward + reverse complement
OUTPUT_DIR  = 'output/predict'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model(CHECKPOINT, CONFIG, device)
window_size = cfg['data']['input_window_length']
genome = GenomeSequence(FASTA)
fa = Fasta(FASTA)
chrom_sizes = {c: len(fa[c]) for c in fa.keys()}
print(f'Model loaded on {device}, window={window_size}bp')

In [ ]:
TRACKS   = ['G1', 'ES', 'MS', 'LS', 'WRT']
records  = {t: [] for t in TRACKS}

# Current model geometry (from RepliformerConfig / dataset constants)
BIN_SIZE  = 128                          # bp per output bin
CROP_BINS = 320                          # bins cropped each side
CROP_BP   = CROP_BINS * BIN_SIZE         # 40960 bp offset to first output bin
OUT_BINS  = 896                          # output bins per window
STRIDE    = OUT_BINS * BIN_SIZE          # 114688 bp — gapless, zero-overlap

if MODE == 'bed':
    regions = list(parse_bed(BED))
else:
    clen = chrom_sizes[CHROM]
    regions = [
        (CHROM, s, s + window_size, f'{CHROM}:{s}-{s+window_size}', '+')
        for s in range(0, clen - window_size + 1, STRIDE)
    ]

print(f'{len(regions):,} regions to predict  (stride={STRIDE} bp, bin={BIN_SIZE} bp)')

In [ ]:
batch_oh, batch_meta = [], []

def flush(batch_oh, batch_meta):
    if not batch_oh:
        return
    x = torch.tensor(np.stack(batch_oh), dtype=torch.float32).to(device)
    if RC_AVERAGE:
        log1p_pred = rc_average(model, x, head=SPECIES).cpu().numpy()  # [B, 896, 4]
    else:
        with torch.no_grad():
            log1p_pred = model(x, head=SPECIES)['rt_signals'].cpu().numpy()

    # channels: G1=0, ES=1, MS=2, LS=3  (SIGNAL_CHANNELS order)
    pred = np.expm1(np.clip(log1p_pred, 0, None))   # [B, 896, 4]  TPM scale

    eps = 1e-6
    g1, es, ms, ls = pred[..., 0], pred[..., 1], pred[..., 2], pred[..., 3]
    wrt = (0.5 * ms + ls) / (es + ms + ls + eps)    # [B, 896]

    for i, (chrom, win_start, _) in enumerate(batch_meta):
        for k in range(OUT_BINS):
            bs = win_start + CROP_BP + k * BIN_SIZE
            be = bs + BIN_SIZE
            for j, t in enumerate(['G1', 'ES', 'MS', 'LS']):
                records[t].append((chrom, bs, be, float(pred[i, k, j])))
            records['WRT'].append((chrom, bs, be, float(wrt[i, k])))

for chrom, start, end, name, strand in regions:
    oh = fetch_one_hot(genome, chrom, start, end, window_size, strand)
    batch_oh.append(oh)
    batch_meta.append((chrom, start, end))
    if len(batch_oh) == BATCH_SIZE:
        flush(batch_oh, batch_meta)
        batch_oh, batch_meta = [], []
flush(batch_oh, batch_meta)

print('Inference done')

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

for track, entries in records.items():
    entries.sort(key=lambda x: (x[0], x[1]))
    bw = pyBigWig.open(str(out_dir / f'{track}.bw'), 'w')
    bw.addHeader(list(chrom_sizes.items()))
    for chrom, start, end, val in entries:
        bw.addEntries([chrom], [start], ends=[end], values=[float(val)])
    bw.close()

print(f'Written {len(TRACKS)} BigWig tracks to {OUTPUT_DIR}')